# Setup and Data Loading:

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scrapbook as sb
import pandas as pd
from utils import *

def get_combined_val_metric_df(agg_df):
    def get_val_metric_df(agg_df, idx=0, postfix=''):
        val_metrics_idx = pd.DataFrame(index=agg_df.index, data=list(agg_df['val_metrics'].apply(lambda x: x[idx])))
        val_metrics_idx.columns = val_metrics_idx.columns.str.replace(f'/dataloader_idx_{idx}', postfix).str.replace('val_', '')
        return val_metrics_idx
    
    agg_df = agg_df[agg_df['val_metrics'].notna()] # TODO: delete this line
    val_metrics_short = get_val_metric_df(agg_df, idx=0)
    val_metrics_long = get_val_metric_df(agg_df, idx=1, postfix='_long')
    val_metrics = val_metrics_short.join(val_metrics_long, how='outer').sort_index(axis=1)
    return val_metrics

# create post-processed aggregate dataframe of all the notebooks' metrics
def load_notebook_df(dir, simple=True):
    book = sb.read_notebooks(dir) # create a scrapbook named `book`
    agg_df = pd.concat([nb.scrap_dataframe for nb in book.notebooks])
    agg_df['experiment_name'] = agg_df['filename'].str.replace('.ipynb', '')
    agg_df = agg_df.pivot(index='experiment_name', columns='name', values='data')
    if simple: agg_df = agg_df.drop(columns=['checkpoint_path', 'uq_over_time'], errors='ignore')
    agg_df = agg_df.apply(lambda col: col.astype(col.dropna().dtype), axis=1) # correct dtypes
    
    agg_df['Bayesian'] = ~agg_df['LPPC_val'].isna() # record if Bayesian model?
    val_metrics = get_combined_val_metric_df(agg_df)
    agg_df = agg_df.drop(columns='val_metrics')
    agg_df = agg_df.join(val_metrics)
    
    return agg_df

In [ ]:
agg_df_VI = load_notebook_df('notebook_runs/VI')
agg_df_MLE = load_notebook_df('notebook_runs/MLE')
agg_df = pd.concat([agg_df_VI, agg_df_MLE])
agg_df.head()

In [ ]:
agg_df.columns # show all columns

# Normalize and Create Summary Metrics:

In [ ]:
# normalize all the columns
normalized_loss_df = agg_df.select_dtypes(include='float').apply(lambda col: np_normalize(col))

# make everything into a "loss"
normalized_loss_df.rename(columns=lambda x: x.replace('LPPC', 'neg_LPPC').replace('R^2', 'neg_R^2'), inplace=True)
maximizer_objectives = normalized_loss_df.columns[normalized_loss_df.columns.str.contains('LPPC') | normalized_loss_df.columns.str.contains('R\^2')]
normalized_loss_df[maximizer_objectives] *= -1 # make everything into a "loss"

print('maximizer_objectives:\n', maximizer_objectives)
print('normalized_loss_df.columns:\n', normalized_loss_df.columns)
display(normalized_loss_df)

## Summary of Findings:
* TS=16 might be the best in general? it's best of Everything_TS=N and Everything_TS=N_E=0.25 comparisons also Everything_TS=16 has the 3rd best energy spectrum, and Everything_TS=16_E=0.25 had really promising Xcor (2/3/26)
* 

In [ ]:
# for average and std, only use important columns
important_loss_names = ['ECE', 'neg_LPPC_val', 'neg_LPPC_val_long', 'mse_bulk_velocity', 'mse_log_energy_spectrum', 'mse_rms', 'mse_self_xcor', 'xcor_MSE_cum', 'xcor_MSE_last',
                        'neg_R^2', 'neg_R^2_long', 'last_TS_neg_R^2', 'last_TS_neg_R^2_long']
important_normalized_loss_df = normalized_loss_df[important_loss_names]

summary_metrics = pd.DataFrame(index=normalized_loss_df.index)
summary_metrics['worst_loss_name'] = normalized_loss_df.apply(lambda row: row.idxmin(skipna=True), axis=1)
summary_metrics['worst_loss'] = normalized_loss_df.apply(lambda row: row.min(skipna=True), axis=1)
summary_metrics['average_loss'] = important_normalized_loss_df.apply(lambda row: row.mean(skipna=True), axis=1)
summary_metrics['loss_std'] = important_normalized_loss_df.apply(lambda row: row.std(skipna=True), axis=1)
summary_metrics = summary_metrics.sort_values(by='worst_loss')
print('='*50+'\nGOTCHA: average and std are only computed on important_loss_names!', flush=True)
print(f'{important_loss_names=}\n{"="*50}')
display(summary_metrics)

In [ ]:
import matplotlib.pyplot as plt
normalized_losses = important_normalized_loss_df.values.flatten()
normalized_losses = normalized_losses[~np.isnan(normalized_losses)]
plt.hist(normalized_losses, bins=100)
plt.locator_params(axis='x', nbins=10)
plt.title('Distribution of important_normalized_losses')
plt.show()

# Is MLE or VI better?
It appears that (at least for this sample) **VI is better at xcor?!** \
(This is not a scientific conclusion! just a guess...)

In [ ]:
print('computed on sample predictions then averaged:')
print(f'VI_xcor_mse = {agg_df_VI["mse_self_xcor"].mean()}')
print(f'MLE_xcor_mse = {agg_df_MLE["mse_self_xcor"].mean()}')

print('\ncomputed on averaged predictions:')
print(f'VI_xcor_mse = {agg_df_VI["xcor_MSE_cum"].mean()}')
print(f'MLE_xcor_mse = {agg_df_MLE["xcor_MSE_cum"].mean()}')

# Looking at Best Experiments:

Unfortunately PP-LL isn't predictive of good flow stats (in particular xcor is bad for Everything_TS=4_gaps_reprod)...

In [ ]:
agg_df_VI.sort_values(by='LPPC_val_long', ascending=False)

The top two are Everything_TS=4_prior=0.05 and Everything_TS=16-E=0.25

In [ ]:
agg_df.sort_values(by='xcor_MSE_cum')

Everything_TS=8_no_PDE has the best mse_log_energy_spectrum but it has quite bad xcor...

In [ ]:
agg_df.sort_values(by='mse_log_energy_spectrum')